# Extratos: normalização, contrapartidas e pendências

Como transformar os movimentos em lançamentos Fênix com origem preservada? Esta análise separa fatos dos arquivos, sugestões contábeis e decisões ainda necessárias. Fontes: Aurora CSV, Meridiano OFX e plano Fênix.

**Limite de negócio:** o documento atualizado de RN-46 a RN-62, D-04 e D-06 não está no workspace. As descrições do pedido orientam perguntas; nenhuma classificação recebe status de regra confirmada. Não há geração dos CSVs finais nem provisões.

In [1]:
from pathlib import Path
from decimal import Decimal
import csv, io, re, hashlib, unicodedata
from collections import Counter
from datetime import datetime
import pandas as pd
from IPython.display import display

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'docs/FORMATO_SAIDA.md').exists())
PASTA = RAIZ / 'Tarvo_Data_Pack_set2026'  # altere para outro pacote; nomes são descobertos abaixo
assert PASTA.is_dir(), f'Pacote ausente: {PASTA}'
def arquivo(padrao):
    encontrados = list(PASTA.glob(padrao))
    assert len(encontrados) == 1, (padrao, encontrados)
    return encontrados[0]
def texto(path):
    bruto = path.read_bytes()
    try: return bruto.decode('utf-8-sig'), 'utf-8-sig'
    except UnicodeDecodeError: return bruto.decode('cp1252'), 'cp1252'
def dinheiro(s):
    s = str(s).strip()
    assert re.fullmatch(r'-?(?:\d{1,3}(?:\.\d{3})+|\d+),\d{2}', s), s
    return Decimal(s.replace('.', '').replace(',', '.'))
def total(valores): return sum(valores, Decimal('0.00'))
def norm(s): return ''.join(c for c in unicodedata.normalize('NFKD', s.upper()) if not unicodedata.combining(c))
pd.set_option('display.max_colwidth', 100)


## 1. Leitura auditável

Aurora: localizar o cabeçalho e registrar a linha física inicial de cada registro CSV, inclusive se houver campo multilinha. Saldo anterior, saldo final, cabeçalhos e avisos são controles, não movimentos. Qualquer outra linha inválida interrompe a leitura. Meridiano: OFX 1.02 SGML, preservando FITID e campos brutos. Plano: código contábil como texto.

In [2]:

A, M, P = arquivo('05a_*.csv'), arquivo('05b_*.ofx'), arquivo('04_*.csv')
hashes = {p:hashlib.sha256(p.read_bytes()).hexdigest() for p in [A,M,P]}
at, ae = texto(A); mt, me = texto(M); pt, pe = texto(P)
display(pd.DataFrame([{'arquivo':p.name,'encoding':e,'bytes':p.stat().st_size,'sha256':hashes[p]} for p,e in [(A,ae),(M,me),(P,pe)]]))
linhas = at.splitlines(keepends=True)
h = next(i for i,l in enumerate(linhas) if l.startswith('Data;Lançamento;'))
reader = csv.reader(io.StringIO(''.join(linhas[h:])), delimiter=';')
colunas = next(reader); aurora = []; controles = []; saldo_inicial = saldo_final = None
while True:
    fisica = h + reader.line_num + 1
    try: campos = next(reader)
    except StopIteration: break
    assert len(campos) == len(colunas), (fisica,campos)
    d = dict(zip(colunas,campos)); desc = d['Lançamento'].strip()
    if desc in ['SALDO ANTERIOR','SALDO FINAL']:
        saldo = dinheiro(d['Saldo (R$)'])
        if desc == 'SALDO ANTERIOR':
            assert saldo_inicial is None
            saldo_inicial = saldo
        else:
            assert saldo_final is None
            saldo_final = saldo
        controles.append({'linha':fisica,'tipo':desc})
    elif not any(v.strip() for v in campos) or desc.startswith('Os lançamentos'):
        controles.append({'linha':fisica,'tipo':'aviso/vazio'})
    else:
        aurora.append(dict(banco='AURORA',ref_origem=f'L{fisica}',
            data=datetime.strptime(d['Data'],'%d/%m/%Y').date(),descricao=d['Lançamento'],
            valor=dinheiro(d['Valor (R$)']), arquivo_origem=A.name,
            saldo=dinheiro(d['Saldo (R$)']), bruto=d))
assert saldo_inicial is not None and saldo_final is not None
blocos = re.findall(r'<STMTTRN>(.*?)</STMTTRN>', mt, re.S)
assert len(blocos) == mt.count('<STMTTRN>') == mt.count('</STMTTRN>')
def tag(bloco, nome):
    valores = re.findall(fr'<{nome}>([^<\r\n]*)',bloco)
    assert len(valores) == 1, (nome,valores)
    return valores[0].strip()
meridiano = []
for bloco in blocos:
    d = {k:tag(bloco,k) for k in ['FITID','DTPOSTED','TRNAMT','MEMO','TRNTYPE','CHECKNUM']}
    assert re.fullmatch(r'-?\d+\.\d{2}', d['TRNAMT'])
    meridiano.append(dict(banco='MERIDIANO',ref_origem=d['FITID'],
        data=datetime.strptime(d['DTPOSTED'][:8],'%Y%m%d').date(),descricao=d['MEMO'],
        valor=Decimal(d['TRNAMT']),arquivo_origem=M.name,bruto=d))
assert tag(mt,'CURDEF') == 'BRL'
plano = list(csv.DictReader(io.StringIO(pt),delimiter=';'))
movimentos = aurora + meridiano
campos_base = ['banco','ref_origem','data','descricao','valor','arquivo_origem']
normalizados = [{k:r[k] for k in campos_base} for r in movimentos]
df = pd.DataFrame(normalizados)
display(df.head()); display(pd.DataFrame(controles))
print('Cabeçalho Aurora:', colunas)
print('Campos OFX:', list(meridiano[0]['bruto']))
print('Campos Fênix:', list(plano[0]))


,arquivo,encoding,bytes,sha256
0,05a_extrato_banco_aurora_cc4471_set2026.csv,utf-8-sig,6896,7303b72fa76d227a863279a36726411605d93fb63237f0cae638c111a70d2d92
1,05b_extrato_banco_meridiano_cc90233_set2026.ofx,cp1252,17458,771a9297c0fa802a677af061fa88a5d30ea5fbe39226a122f87cebdb69f17e9f
2,04_plano_de_contas_fenix_erp.csv,cp1252,3152,8f7c8c0abe1599ba434a7e99fb2997447f4a11142fee7c131babd79820175e43


,banco,ref_origem,data,descricao,valor,arquivo_origem
0,AURORA,L9,2026-09-01,TED RECEBIDA 99.876.669/0001-09 ORBE ADQUIRENCIA SA,66026.00,05a_extrato_banco_aurora_cc4471_set2026.csv
1,AURORA,L10,2026-09-01,TAR PACOTE SERVICOS PJ,-389.00,05a_extrato_banco_aurora_cc4471_set2026.csv
2,AURORA,L11,2026-09-02,TED RECEBIDA 99.876.669/0001-09 ORBE ADQUIRENCIA SA,209630.03,05a_extrato_banco_aurora_cc4471_set2026.csv
3,AURORA,L12,2026-09-02,TRANSF ENTRE CONTAS P/ BCO MERIDIANO AG 0091 CC 90233-5,-350000.00,05a_extrato_banco_aurora_cc4471_set2026.csv
4,AURORA,L13,2026-09-03,TED RECEBIDA 99.876.669/0001-09 ORBE ADQUIRENCIA SA,170754.70,05a_extrato_banco_aurora_cc4471_set2026.csv


,linha,tipo
0,8,SALDO ANTERIOR
1,93,aviso/vazio
2,94,SALDO FINAL
3,95,aviso/vazio


Cabeçalho Aurora: ['Data', 'Lançamento', 'Documento', 'Valor (R$)', 'Saldo (R$)']
Campos OFX: ['FITID', 'DTPOSTED', 'TRNAMT', 'MEMO', 'TRNTYPE', 'CHECKNUM']
Campos Fênix: ['COD_CONTA', 'DESCRICAO', 'TIPO', 'NATUREZA', 'ACEITA_LANCTO', 'COD_SUPERIOR', 'DT_CRIACAO']


## 2. Qualidade e conservação financeira

Duplicidade de conteúdo é um alerta; duplicidade de referência compromete a identidade. Não excluir registros. Data bancária indica movimentação de caixa, enquanto competência pode estar no histórico ou exigir documento externo. O saldo inicial do Meridiano é somente implícito: não há saldo anterior independente para validá-lo.

In [3]:

resumos = []
for banco, base in [('AURORA',aurora),('MERIDIANO',meridiano)]:
    vals=[r['valor'] for r in base]
    chaves=Counter(r['ref_origem'] for r in base)
    conteudos=Counter((r['data'],r['descricao'],r['valor']) for r in base)
    resumos.append(dict(banco=banco,movimentos=len(base),entradas=total(v for v in vals if v>0),
        saidas=total(v for v in vals if v<0),liquido=total(vals),positivos=sum(v>0 for v in vals),
        negativos=sum(v<0 for v in vals),zeros=vals.count(Decimal(0)),
        referencias_repetidas=sum(n-1 for n in chaves.values()),
        conteudos_repetidos=sum(n-1 for n in conteudos.values()),
        inicio=min(r['data'] for r in base),fim=max(r['data'] for r in base)))
display(pd.DataFrame(resumos))
display(df.apply(lambda s:s.map(lambda x:x is None or x == '').sum()).to_frame('ausentes'))
print('Documento Aurora ausente:',sum(not r['bruto']['Documento'] for r in aurora))
saldo=saldo_inicial; divergencias=[]
for r in aurora:
    saldo += r['valor']
    if saldo != r['saldo']: divergencias.append(r['ref_origem'])
print('Saldos Aurora divergentes:',divergencias)
assert not divergencias and saldo == saldo_final
print('Aurora: inicial',saldo_inicial,'+ movimentos',total(r['valor'] for r in aurora),'= final',saldo_final)
final_m=Decimal(tag(mt,'BALAMT')); inicial_m=final_m-total(r['valor'] for r in meridiano)
print('Meridiano: final',final_m,'; inicial IMPLÍCITO',inicial_m)
print('Linhas Aurora com saldo negativo:',sum(r['saldo']<0 for r in aurora))
inconsistentes=[r['ref_origem'] for r in meridiano if (r['bruto']['TRNTYPE']=='CREDIT' and r['valor']<=0) or (r['bruto']['TRNTYPE']=='DEBIT' and r['valor']>=0)]
assert not inconsistentes, inconsistentes
print('OFX: período declarado',tag(mt,'DTSTART'),tag(mt,'DTEND'))
assert all(tag(mt,'DTSTART')[:8] <= r['bruto']['DTPOSTED'][:8] <= tag(mt,'DTEND')[:8] for r in meridiano)
df['competencia_no_historico'] = df.descricao.str.extract(r'(\d{2}/\d{4})',expand=False)
display(df[df.competencia_no_historico.notna()][['ref_origem','data','descricao','competencia_no_historico']])


,banco,movimentos,entradas,saidas,liquido,positivos,negativos,zeros,referencias_repetidas,conteudos_repetidos,inicio,fim
0,AURORA,84,3576996.29,-6109419.70,-2532423.41,23,61,0,0,0,2026-09-01,2026-09-30
1,MERIDIANO,100,7318364.90,-6202029.67,1116335.23,72,28,0,0,0,2026-09-01,2026-09-30


,ausentes
banco,0
ref_origem,0
data,0
descricao,0
valor,0
arquivo_origem,0


Documento Aurora ausente: 63
Saldos Aurora divergentes: []
Aurora: inicial 612334.81 + movimentos -2532423.41 = final -1920088.60
Meridiano: final 1320453.26 ; inicial IMPLÍCITO 204118.03
Linhas Aurora com saldo negativo: 62
OFX: período declarado 20260901000000[-3:BRT] 20260930235959[-3:BRT]


,ref_origem,data,descricao,competencia_no_historico
0,L9,2026-09-01,TED RECEBIDA 99.876.669/0001-09 ORBE ADQUIRENCIA SA,69/0001
2,L11,2026-09-02,TED RECEBIDA 99.876.669/0001-09 ORBE ADQUIRENCIA SA,69/0001
4,L13,2026-09-03,TED RECEBIDA 99.876.669/0001-09 ORBE ADQUIRENCIA SA,69/0001
5,L14,2026-09-04,TED RECEBIDA 99.876.669/0001-09 ORBE ADQUIRENCIA SA,69/0001
6,L15,2026-09-04,PAGTO SALARIOS - FOLHA 08/2026,08/2026
8,L17,2026-09-08,TED RECEBIDA 99.876.669/0001-09 ORBE ADQUIRENCIA SA,69/0001
16,L25,2026-09-09,TED RECEBIDA 99.876.669/0001-09 ORBE ADQUIRENCIA SA,69/0001
22,L31,2026-09-10,TED RECEBIDA 99.876.669/0001-09 ORBE ADQUIRENCIA SA,69/0001
33,L42,2026-09-10,PAGTO ISS SAO PAULO DAMSP 08/2026,08/2026
34,L43,2026-09-11,TED RECEBIDA 99.876.669/0001-09 ORBE ADQUIRENCIA SA,69/0001


## 3. Padrões e contas candidatas

Os padrões abaixo são hipóteses explícitas de contrapartida. O sinal esperado faz parte do reconhecimento. Aplicação e resgate não devem virar receita integral. Nomes truncados não identificam lojista. Uma conta existente e analítica é condição necessária, mas não confirma a regra.

In [4]:

# natureza, padrão, sinal esperado, conta candidata, referência a validar
padroes = [
('transferencia_propria',r'TRANSF ENTRE CONTAS|MESMA TITULARIDADE',None,'1.1.09.001','RN-54'),
('credito_nao_identificado',r'SEM IDENTIFICACAO',1,'1.1.09.002','RN-52'),
('devolucao_pix',r'DEVOLUCAO PIX',1,'2.1.01.002','RN-53 / D-06'),
('comissao_anterior',r'PONTE CAMBIO SA REPASSE COMISSAO',1,'1.1.03.002','RN-49'),
('recebimento_orbe',r'TED RECEBIDA .*ORBE',1,'1.1.03.001','RN-56'),
('salarios',r'PAGTO SALARIOS',-1,'2.1.04.001','RN-47'),
('inss',r'GPS INSS',-1,'2.1.04.002','RN-47'),
('fgts',r'GRF FGTS',-1,'2.1.04.003','RN-47'),
('iss',r'PAGTO ISS',-1,'2.1.03.003','RN-47'),
('irrf',r'DARF .* IRRF',-1,'2.1.03.004','RN-47'),
('pis',r'DARF .* PIS',-1,'2.1.03.001','RN-47'),
('cofins',r'DARF .* COFINS',-1,'2.1.03.002','RN-47'),
('tarifas',r'^TAR(?: |IFA)',-1,'4.2.01','RN-50'),
('aluguel_condominio',r'BOLETO CONDOMINIO',-1,'4.2.02','RN-50'),
('cloud',r'AMAZON WEB SERVICES',-1,'4.2.03','RN-50'),
('assinaturas',r'GOOGLE WORKSPACE',-1,'4.2.04','RN-50'),
('servicos',r'CONTABILIDADE EXATA',-1,'4.2.06','RN-50'),
('repasse_lojista',r'^PIX ENVIADO (?!CONTABILIDADE EXATA)',-1,'2.1.01.001','validar obrigação'),
('recurso_remessa',r'^PIX RECEB',1,'2.1.02.001','RN-55: conta a confirmar'),
('liquidacao_ponte',r'PONTE CAMBIO SA LIQ',-1,'2.1.02.001','validar liquidação'),
('aplicacao',r'^APLICACAO AUTOMATICA',-1,'1.1.02.001','validar investimento'),
('resgate',r'^RESGATE AUTOMATICO',1,'1.1.02.001','validar composição do resgate'),
('rendimento',r'^RENDIMENTO APLIC',1,'3.2.01','validar rendimento'),
('seguro',r'^SEGURO',-1,'4.2.05','validar competência/provisão'),
('iof',r'^IOF',-1,'4.3.01','validar natureza/provisão')]
por_codigo={p['COD_CONTA']:p for p in plano}
assert len(por_codigo)==len(plano)
problemas=[]
for p in plano:
    pai=p['COD_SUPERIOR']
    if pai and (pai not in por_codigo or not p['COD_CONTA'].startswith(pai+'.')): problemas.append(p)
    if (p['TIPO']=='ANALITICA') != (p['ACEITA_LANCTO']=='S'): problemas.append(p)
assert not problemas
print('Contas:',len(plano),'| aceitam lançamento:',sum(p['ACEITA_LANCTO']=='S' for p in plano))
display(pd.DataFrame(plano))
for r in normalizados:
    hits=[p for p in padroes if re.search(p[1],norm(r['descricao']))]
    # PIX sem identificação é mais específico que o padrão genérico de recebimento.
    if any(p[0]=='credito_nao_identificado' for p in hits): hits=[p for p in hits if p[0]=='credito_nao_identificado']
    assert len(hits)<=1, (r,hits)
    p=hits[0] if hits else None
    sinal_ok=bool(p and (p[2] is None or (r['valor']>0 if p[2]==1 else r['valor']<0)))
    r.update(natureza=p[0] if p else 'nao_reconhecido',
        conta_candidata=p[3] if sinal_ok else '',regra_candidata=p[4] if p else '',
        status='HIPOTESE',sinal_ok=sinal_ok)
    if r['conta_candidata']:
        assert por_codigo[r['conta_candidata']]['ACEITA_LANCTO']=='S'
        assert r['conta_candidata'] not in ['1.1.01.001','1.1.01.002']
ndf=pd.DataFrame(normalizados)
display(ndf.groupby(['banco','natureza','conta_candidata'],dropna=False).agg(quantidade=('valor','size'),valor=('valor',total)))


Contas: 52 | aceitam lançamento: 32


,COD_CONTA,DESCRICAO,TIPO,NATUREZA,ACEITA_LANCTO,COD_SUPERIOR,DT_CRIACAO
0,1,ATIVO,SINTETICA,D,N,,15/01/2024
1,1.1,ATIVO CIRCULANTE,SINTETICA,D,N,1,15/01/2024
2,1.1.01,DISPONIBILIDADES,SINTETICA,D,N,1.1,15/01/2024
3,1.1.01.001,Banco Aurora C/C 4471-0,ANALITICA,D,S,1.1.01,15/01/2024
4,1.1.01.002,Banco Meridiano C/C 90233-5 (Câmbio),ANALITICA,D,S,1.1.01,15/01/2024
5,1.1.02,APLICAÇÕES FINANCEIRAS,SINTETICA,D,N,1.1,15/01/2024
6,1.1.02.001,Aplicação automática Meridiano,ANALITICA,D,S,1.1.02,15/01/2024
7,1.1.03,CRÉDITOS A RECEBER,SINTETICA,D,N,1.1,15/01/2024
8,1.1.03.001,Recebíveis de adquirente - Orbe,ANALITICA,D,S,1.1.03,15/01/2024
9,1.1.03.002,Comissões de intermediação a receber - câmbio,ANALITICA,D,S,1.1.03,15/01/2024


quantidade        valor
banco     natureza                 conta_candidata                         
AURORA    aluguel_condominio       4.2.02                    1    -28450.00
          assinaturas              4.2.04                    1     -2210.00
          cloud                    4.2.03                    1    -41208.77
          cofins                   2.1.03.002                1    -45282.46
          credito_nao_identificado 1.1.09.002                1      7350.00
          devolucao_pix            2.1.01.002                1     12400.00
          fgts                     2.1.04.003                1    -14778.57
          inss                     2.1.04.002                1    -51320.66
          irrf                     2.1.03.004                1    -22104.35
          iss                      2.1.03.003                1    -21877.40
          pis                      2.1.03.001                1     -9811.20
          recebimento_orbe         1.1.03.001               21   3557246.29
          repasse_lojista          2.1.01.001               44  -4230267.61
          salarios                 2.1.04.001                1   -184732.18
          servicos                 4.2.06                    1     -6800.00
          tarifas                  4.2.01                    2      -576.50
          transferencia_propria    1.1.09.001                4  -1450000.00
MERIDIANO aplicacao                1.1.02.001                3   -588124.00
          comissao_anterior        1.1.03.002                1     61884.12
          iof                      4.3.01                    1       -48.21
          liquidacao_ponte         2.1.02.001               21  -5611447.12
          recurso_remessa          2.1.02.001               64   5629687.12
          rendimento               3.2.01                    1      3912.66
          resgate                  1.1.02.001                3    472881.00
          seguro                   4.2.05                    1     -1830.44
          tarifas                  4.2.01                    2      -579.90
          transferencia_propria    1.1.09.001                3   1150000.00

## 4. Transferências e exceções

Candidatos a par exigem históricos compatíveis, mesma data, sinais opostos e valor exato, com unicidade nos dois lados. Isso fornece evidência bancária, sem aprovação contábil. Sem par na janela, investigar trânsito; não declarar perda nem presumir chegada no mês seguinte.

In [5]:

saidas=[r for r in normalizados if r['banco']=='AURORA' and r['natureza']=='transferencia_propria']
entradas=[r for r in normalizados if r['banco']=='MERIDIANO' and r['natureza']=='transferencia_propria']
pares=[]
for s in saidas:
    candidatos=[e for e in entradas if e['data']==s['data'] and e['valor']==-s['valor']]
    unico=len(candidatos)==1 and sum(x['data']==s['data'] and x['valor']==s['valor'] for x in saidas)==1
    pares.append(dict(ref_aurora=s['ref_origem'],data=s['data'],valor=s['valor'],
        ref_meridiano=candidatos[0]['ref_origem'] if unico else '',status='par candidato' if unico else 'investigar trânsito/ambiguidade'))
display(pd.DataFrame(pares))
print('Saldo líquido das transferências:',total(r['valor'] for r in normalizados if r['natureza']=='transferencia_propria'))
# Sem documento atualizado, todos os movimentos aguardam liberação da classificação.
pendencias=[]
for r in normalizados:
    motivo='Documento atualizado ausente; contrapartida e regra ainda não aprovadas.'
    passo='Obter versão das regras e validar padrão, sinal e contrapartida com financeiro.'
    if r['natureza']=='credito_nao_identificado':
        motivo+=' Histórico contém SEM IDENTIFICACAO, mesmo exibindo número de documento.'
        passo='Identificar pagador por comprovante e validar conta transitória.'
    elif r['natureza']=='devolucao_pix':
        motivo+=' Devolução não comprova extinção da obrigação com lojista.'
        passo='Solicitar pagamento original e confirmar obrigação ainda devida.'
    elif r['natureza']=='recurso_remessa':
        motivo+=' Nome truncado não comprova cadastro nem destinação de remessa.'
        passo='Obter identificador/comprovante; comercial decide cadastro/contrato, financeiro classifica.'
    elif r['natureza']=='transferencia_propria':
        passo='Revisar pares acima; buscar extrato subsequente para itens sem par.'
    elif r['natureza']=='resgate':
        passo='Obter extrato da aplicação para separar principal, rendimento e impostos.'
    pendencias.append(dict(tipo='VALIDACAO_CLASSIFICACAO',banco=r['banco'],ref_origem=r['ref_origem'],
        arquivo_origem=r['arquivo_origem'],valor_brl=r['valor'],motivo=motivo,proximo_passo=passo,quem_decide='FINANCEIRO'))
display(pd.DataFrame(pendencias))
print('Classificações aprovadas: 0; pendências:',len(pendencias))


,ref_aurora,data,valor,ref_meridiano,status
0,L12,2026-09-02,-350000.00,MRD2026090107,par candidato
1,L44,2026-09-11,-420000.00,MRD2026090274,par candidato
2,L58,2026-09-21,-380000.00,MRD2026090454,par candidato
3,L92,2026-09-30,-300000.00,,investigar trânsito/ambiguidade


Saldo líquido das transferências: -300000.00


,tipo,banco,ref_origem,arquivo_origem,valor_brl,motivo,proximo_passo,quem_decide
0,VALIDACAO_CLASSIFICACAO,AURORA,L9,05a_extrato_banco_aurora_cc4471_set2026.csv,66026.00,Documento atualizado ausente; contrapartida e regra ainda não aprovadas.,"Obter versão das regras e validar padrão, sinal e contrapartida com financeiro.",FINANCEIRO
1,VALIDACAO_CLASSIFICACAO,AURORA,L10,05a_extrato_banco_aurora_cc4471_set2026.csv,-389.00,Documento atualizado ausente; contrapartida e regra ainda não aprovadas.,"Obter versão das regras e validar padrão, sinal e contrapartida com financeiro.",FINANCEIRO
2,VALIDACAO_CLASSIFICACAO,AURORA,L11,05a_extrato_banco_aurora_cc4471_set2026.csv,209630.03,Documento atualizado ausente; contrapartida e regra ainda não aprovadas.,"Obter versão das regras e validar padrão, sinal e contrapartida com financeiro.",FINANCEIRO
3,VALIDACAO_CLASSIFICACAO,AURORA,L12,05a_extrato_banco_aurora_cc4471_set2026.csv,-350000.00,Documento atualizado ausente; contrapartida e regra ainda não aprovadas.,Revisar pares acima; buscar extrato subsequente para itens sem par.,FINANCEIRO
4,VALIDACAO_CLASSIFICACAO,AURORA,L13,05a_extrato_banco_aurora_cc4471_set2026.csv,170754.70,Documento atualizado ausente; contrapartida e regra ainda não aprovadas.,"Obter versão das regras e validar padrão, sinal e contrapartida com financeiro.",FINANCEIRO
...,...,...,...,...,...,...,...,...
179,VALIDACAO_CLASSIFICACAO,MERIDIANO,MRD2026090670,05b_extrato_banco_meridiano_cc90233_set2026.ofx,146817.00,Documento atualizado ausente; contrapartida e regra ainda não aprovadas.,"Obter extrato da aplicação para separar principal, rendimento e impostos.",FINANCEIRO
180,VALIDACAO_CLASSIFICACAO,MERIDIANO,MRD2026090674,05b_extrato_banco_meridiano_cc90233_set2026.ofx,-240282.30,Documento atualizado ausente; contrapartida e regra ainda não aprovadas.,"Obter versão das regras e validar padrão, sinal e contrapartida com financeiro.",FINANCEIRO
181,VALIDACAO_CLASSIFICACAO,MERIDIANO,MRD2026090680,05b_extrato_banco_meridiano_cc90233_set2026.ofx,3912.66,Documento atualizado ausente; contrapartida e regra ainda não aprovadas.,"Obter versão das regras e validar padrão, sinal e contrapartida com financeiro.",FINANCEIRO
182,VALIDACAO_CLASSIFICACAO,MERIDIANO,MRD2026090688,05b_extrato_banco_meridiano_cc90233_set2026.ofx,-141194.63,Documento atualizado ausente; contrapartida e regra ainda não aprovadas.,"Obter versão das regras e validar padrão, sinal e contrapartida com financeiro.",FINANCEIRO


Classificações aprovadas: 0; pendências: 184


## 5. Matriz de regras a validar

Cada linha informa evidência disponível, conta candidata e teste necessário. Todas aguardam o documento atualizado. RN-58 a RN-62 não têm descrição individual no material recebido; não é possível atribuir requisitos específicos a cada número.

In [6]:

matriz = [
('RN-46','descricao, sinal, banco','todos','contrapartida, não conta bancária','comparar evento e conta analítica','histórico ambíguo'),
('RN-47','descricao, competência no histórico','folha e tributos','passivos conforme tabela de padrões','obter razão/provisão e confirmar baixa','provisão não comprovada'),
('RN-48','ausência de fonte de provisão','nenhum evento gerado','não gerar provisão','verificar que saídas só derivam de movimentos','regra completa ausente'),
('RN-49','MEMO REPASSE COMISSAO, data, valor','comissao_anterior','1.1.03.002 candidata','obter razão/comissão da competência anterior','crédito não comprova saldo a receber'),
('RN-50','historico e sinal negativo','tarifas/aluguel/cloud/assinaturas/serviços','4.2.01/02/03/04/06 candidatas','validar despesa do mês com documentação','despesa antecipada ou provisionada'),
('RN-51','ACEITA_LANCTO, COD_CONTA','toda conta candidata','somente S','assert de existência e aceitação','conta sintética/inativa'),
('RN-52','histórico SEM IDENTIFICACAO','credito_nao_identificado','1.1.09.002 candidata','exigir pendência e comprovação de pagador','número no histórico não resolve identidade'),
('RN-53 / D-06','DEVOLUCAO PIX, sinal','devolucao_pix','2.1.01.002 candidata','ligar pagamento original e obrigação','devolução sem chave de pagamento'),
('RN-54','data, sinais, históricos, valor','transferencia_propria','1.1.09.001 candidata','unicidade dos pares e trânsito na data de corte','par fora da janela ou múltiplos candidatos'),
('RN-55','PIX RECEB, MEMO, FITID','recurso_remessa','conta definida pela regra ainda desconhecida','obter comprovante e destinação','não inferir adiantamento só pelo nome'),
('RN-56','histórico Orbe, valor agregado','recebimento_orbe','1.1.03.001 candidata','não vincular parcelas individuais','igualdade agregada não prova composição'),
('RN-57 / D-04','linha física, FITID','todos','não se aplica','conferir referências contra arquivos originais','FITID vazio/duplicado'),
('RN-58 a RN-62','responsabilidades, aprovações, pendências','fluxo de revisão','não se aplica','obter texto individual e registrar aprovação/fechamento parcial','requisitos individuais ausentes')]
display(pd.DataFrame(matriz,columns=['regra','campos','padrao','conta','verificacao','excecao']).assign(status='AGUARDA_DOCUMENTO'))


,regra,campos,padrao,conta,verificacao,excecao,status
0,RN-46,"descricao, sinal, banco",todos,"contrapartida, não conta bancária",comparar evento e conta analítica,histórico ambíguo,AGUARDA_DOCUMENTO
1,RN-47,"descricao, competência no histórico",folha e tributos,passivos conforme tabela de padrões,obter razão/provisão e confirmar baixa,provisão não comprovada,AGUARDA_DOCUMENTO
2,RN-48,ausência de fonte de provisão,nenhum evento gerado,não gerar provisão,verificar que saídas só derivam de movimentos,regra completa ausente,AGUARDA_DOCUMENTO
3,RN-49,"MEMO REPASSE COMISSAO, data, valor",comissao_anterior,1.1.03.002 candidata,obter razão/comissão da competência anterior,crédito não comprova saldo a receber,AGUARDA_DOCUMENTO
4,RN-50,historico e sinal negativo,tarifas/aluguel/cloud/assinaturas/serviços,4.2.01/02/03/04/06 candidatas,validar despesa do mês com documentação,despesa antecipada ou provisionada,AGUARDA_DOCUMENTO
5,RN-51,"ACEITA_LANCTO, COD_CONTA",toda conta candidata,somente S,assert de existência e aceitação,conta sintética/inativa,AGUARDA_DOCUMENTO
6,RN-52,histórico SEM IDENTIFICACAO,credito_nao_identificado,1.1.09.002 candidata,exigir pendência e comprovação de pagador,número no histórico não resolve identidade,AGUARDA_DOCUMENTO
7,RN-53 / D-06,"DEVOLUCAO PIX, sinal",devolucao_pix,2.1.01.002 candidata,ligar pagamento original e obrigação,devolução sem chave de pagamento,AGUARDA_DOCUMENTO
8,RN-54,"data, sinais, históricos, valor",transferencia_propria,1.1.09.001 candidata,unicidade dos pares e trânsito na data de corte,par fora da janela ou múltiplos candidatos,AGUARDA_DOCUMENTO
9,RN-55,"PIX RECEB, MEMO, FITID",recurso_remessa,conta definida pela regra ainda desconhecida,obter comprovante e destinação,não inferir adiantamento só pelo nome,AGUARDA_DOCUMENTO


## 6. Verificações e evidências

Os testes abaixo validam normalização e rastreabilidade; não validam decisões contábeis. As somas usam Decimal. A trilha mantém registros brutos e hash das entradas.

In [7]:

checks=[]
def verificar(nome,condicao,evidencia):
    checks.append({'verificacao':nome,'ok':bool(condicao),'evidencia':str(evidencia)})
    assert condicao, (nome,evidencia)
verificar('quantidade',len(normalizados)==len(aurora)+len(blocos),len(normalizados))
for banco,base in [('AURORA',aurora),('MERIDIANO',meridiano)]:
    normal=[r for r in normalizados if r['banco']==banco]
    verificar(banco+' valores individuais',[(r['ref_origem'],r['valor']) for r in base]==[(r['ref_origem'],r['valor']) for r in normal],len(base))
    verificar(banco+' soma',total(r['valor'] for r in base)==total(r['valor'] for r in normal),total(r['valor'] for r in normal))
    verificar(banco+' referencias únicas',len({r['ref_origem'] for r in base})==len(base),len(base))
    verificar(banco+' referências presentes',all(r['ref_origem'] for r in base),len(base))
for r in aurora:
    i=int(r['ref_origem'][1:])-1
    verificar('origem '+r['ref_origem'],r['bruto']['Data'] in linhas[i],r['ref_origem'])
verificar('FITID original',[r['ref_origem'] for r in meridiano]==[tag(b,'FITID') for b in blocos],len(blocos))
verificar('Decimal sem perda',all(isinstance(r['valor'],Decimal) for r in normalizados),len(normalizados))
verificar('cobertura de pendências',{(r['arquivo_origem'],r['ref_origem']) for r in normalizados}=={(r['arquivo_origem'],r['ref_origem']) for r in pendencias},len(pendencias))
verificar('entradas preservadas',all(hashlib.sha256(p.read_bytes()).hexdigest()==hashes[p] for p in hashes),len(hashes))
verificar('sem descarte Aurora',len(aurora)+len(controles)==len(list(csv.reader(io.StringIO(''.join(linhas[h+1:])),delimiter=';'))),len(aurora)+len(controles))
verificar('sem aprovação implícita',all(r['status']=='HIPOTESE' for r in normalizados),len(normalizados))
display(pd.DataFrame(checks))


,verificacao,ok,evidencia
0,quantidade,True,184
1,AURORA valores individuais,True,84
2,AURORA soma,True,-2532423.41
3,AURORA referencias únicas,True,84
4,AURORA referências presentes,True,84
...,...,...,...
94,Decimal sem perda,True,184
95,cobertura de pendências,True,184
96,entradas preservadas,True,3
97,sem descarte Aurora,True,88


## 7. Contrato e efeito nos outputs

Proposta: `classificar_extratos(pasta, regras) -> (lancamentos, pendencias)`. O parâmetro `regras` precisa carregar versão, status de aprovação, evidência, vigência, padrão, sinal e contrapartida. Regra ausente ou ambígua produz pendência. A função não foi implementada nesta etapa.

| Retorno | Campos necessários |
|---|---|
| lançamentos aprovados | id_saida, banco, ref_origem, data, descricao, valor (Decimal), conta_codigo, metodo, confianca, arquivo_origem, regra_id, versao_regra, evidencias |
| pendências | id_saida, tipo, banco, ref_origem, valor_brl (Decimal), motivo, proximo_passo, quem_decide, arquivo_origem, status, regra_id |

`lancamentos_classificados.csv` recebe apenas classificações aprovadas; contas candidatas ficam em revisão. A cobertura deve contabilizar **classificados ∪ pendentes**, admitindo um movimento classificado com pendência obrigatória (crédito não identificado, por exemplo). O status contábil e a liberação do fechamento são dimensões distintas.

`pendencias.csv` conserva as colunas exigidas em `docs/FORMATO_SAIDA.md` e acrescenta próximo passo, banco e arquivo. `quem_decide` segue a responsabilidade: comercial para cadastro/taxa/contrato; financeiro para classificação/conciliação/liberação. O encaminhamento detalhado aguarda RN-58 a RN-62.

`rastro.csv` pode ser derivado das referências incluídas nos retornos. Para cada id_saida, a soma dos valores das origens deve igualar o valor da saída. IDs devem ser determinísticos e incluir identidade de arquivo/conta/período e referência; L9 isoladamente não é globalmente único. Para resultados agregados, registrar cada contribuição assinada; não repetir o total em todas as linhas do rastro.

Serialização: CSV com `;`, Decimal formatado com duas casas e ponto, data ISO, sem converter monetários para float. `metodo=REGRA` exige regra aprovada; `HUMANO` exige registro da decisão. Não atribuir confiança numérica fictícia às hipóteses. Não gerar os arquivos finais a partir deste estudo.

In [8]:

print('Movimentos analisados:',len(normalizados))
print('Com contrapartida candidata:',sum(bool(r['conta_candidata']) for r in normalizados))
print('Sem padrão/contrapartida segura:',sum(not r['conta_candidata'] for r in normalizados))
print('Aprovados para saída final: 0 (documento de regras ausente).')
print('Transferências sem par candidato:',sum(not p['ref_meridiano'] for p in pares))
display(ndf[ndf.natureza.isin(['credito_nao_identificado','devolucao_pix','comissao_anterior'])][['banco','ref_origem','descricao','valor','conta_candidata']])


Movimentos analisados: 184
Com contrapartida candidata: 184
Sem padrão/contrapartida segura: 0
Aprovados para saída final: 0 (documento de regras ausente).
Transferências sem par candidato: 1


,banco,ref_origem,descricao,valor,conta_candidata
41,AURORA,L50,PIX RECEBIDO 38686002000180 SEM IDENTIFICACAO,7350.00,1.1.09.002
56,AURORA,L65,DEVOLUCAO PIX - MUNDO ABERTO INTERC,12400.00,2.1.01.002
102,MERIDIANO,MRD2026090132,TED RECEB PONTE CAMBIO SA REPASSE COMISSAO 08/2026,61884.12,1.1.03.002


## Conclusões para implementação

A leitura precisa ser específica por formato: Aurora inclui saldos e avisos, enquanto o Meridiano usa SGML e fornece FITID. Documentos bancários ausentes no Aurora não impedem rastreabilidade por linha. O saldo negativo é uma condição a investigar com o financeiro, não erro de parsing. O saldo inicial implícito do Meridiano não constitui conciliação independente.

Os padrões permitem propor contrapartidas para tarifas, serviços, folha, tributos, recebíveis e remessas. Automatização depende de validar essas propostas contra as regras atualizadas. Folha e tributos não podem virar despesa apenas pelo pagamento; resgate não pode virar receita integral. Um valor coincidente não confirma regra nem identidade.

As tabelas anteriores mostram os pares de transferências, o item em possível trânsito, o crédito sem identificação, a devolução e o recebimento de comissão anterior. Esses casos precisam de comprovantes, razão ou decisão humana. Nomes truncados permanecem sem vínculo automático a lojista.

Em `src/`, próxima etapa: leitores que preservem referência e bruto; normalização Decimal; validação do plano; motor determinístico com regras versionadas; pendências com aprovação; exportadores dos três CSVs com validação do rastro. Manter o fechamento parcial explícito e não incluir provisões nesta função. Nenhum módulo de produção foi alterado.